In [1]:
import os
import sys
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

# Permitir que el notebook lea el archivo agent.py de la raíz
if '..' not in sys.path:
    sys.path.append('..')

from agent import pregunta_a_sql

# Cargamos credenciales del entorno
load_dotenv()

def testear_agente(pregunta_usuario: str):
    """Función optimizada para escupir los tests en ráfaga sin warnings."""
    try:
        # 1. Generar SQL con la IA
        sql = pregunta_a_sql(pregunta_usuario)
        print(f"❓ PREGUNTA: {pregunta_usuario}")
        print(f"💻 SQL GENERADO:\n{sql}\n")
        
        # 2. Conexión limpia con SQLAlchemy
        usuario = os.getenv("DB_USER")
        password = os.getenv("DB_PASSWORD")
        host = os.getenv("DB_HOST", "localhost")
        port = os.getenv("DB_PORT", "3306")
        db_name = os.getenv("DB_NAME")
        
        engine = create_engine(f"mysql+mysqlconnector://{usuario}:{password}@{host}:{port}/{db_name}")
        
        # 3. Ejecutar y mostrar
        df = pd.read_sql(sql, engine)
        if df.empty:
            print("📊 RESULTADO: Consulta ejecutada con éxito (0 registros devueltos).")
        else:
            print("📊 RESULTADO:")
            display(df) # display() hace que en Jupyter la tabla se vea nativa y hermosa
            
    except Exception as error:
        print(f"❌ ERROR EN ESTA CONSULTA: {error}")
    
    print("-" * 80 + "\n")

In [2]:
# Lista de las 10+ preguntas clave del sistema de la biblioteca
bateria_preguntas = [
    # --- 1. Consultas de Verificación Básica ---
    "¿Cuáles son los títulos de los libros que tenemos en la biblioteca?",
    "Mostrame el nombre, apellido y email de todos los socios activos.",
    "Listar los autores de nacionalidad Argentina.",
    
    # --- 2. Consultas con Filtros y Agrupaciones ---
    "¿Cuántos libros hay de cada género en la biblioteca?",
    "Mostrame los títulos de los libros escritos por el autor Isaac Asimov.",
    "¿Qué libros fueron publicados después del año 2020?",
    
    # --- 3. Consultas usando tus Vistas de Reportes ---
    "¿Qué libros están por agotarse?", 
    "Mostrame los socios morosos",
    
    # --- 4. Consultas de Reglas de Negocio Complejas ---
    "¿Cuáles son los socios que actualmente se encuentran suspendidos?",
    "Mostrame los préstamos que están vencidos actualmente.",
    
    # --- 5. La Prueba de Fuego (Lógica Temporal Compleja) ---
    "Aquellos libros para los que existe más de un ejemplar, tal que al menos dos de esos ejemplares se hayan encontrado prestados en forma simultánea en un determinado momento. Para simplificar, considerar solamente aquellos préstamos en los que el libro ya haya sido devuelto."
]

print(f"🚀 Iniciando ejecución automática de los {len(bateria_preguntas)} casos de prueba...\n")

# Ejecutamos cada pregunta de la lista usando nuestra función automatizada
for i, pregunta in enumerate(bateria_preguntas, 1):
    print(f"=== CASO DE PRUEBA #{i} ===")
    testear_agente(pregunta)

🚀 Iniciando ejecución automática de los 11 casos de prueba...

=== CASO DE PRUEBA #1 ===
❓ PREGUNTA: ¿Cuáles son los títulos de los libros que tenemos en la biblioteca?
💻 SQL GENERADO:
SELECT titulo FROM libro

📊 RESULTADO:


,titulo
0,Y no quedó ninguno
1,Asesinato en el Orient Express
2,Sapiens: De animales a dioses
3,El Señor de los Anillos: La Comunidad del Anillo
4,El Señor de los Anillos: Las Dos Torres
5,El Señor de los Anillos: El Retorno del Rey
6,Cien años de soledad
7,El Resplandor
8,Ficciones
9,Cosmos


--------------------------------------------------------------------------------

=== CASO DE PRUEBA #2 ===
❓ PREGUNTA: Mostrame el nombre, apellido y email de todos los socios activos.
💻 SQL GENERADO:
SELECT nombre, apellido, email FROM socio WHERE estado = 'ACTIVO'

📊 RESULTADO:


,nombre,apellido,email
0,Lucas,Pérez,lucas@gmail.com
1,María,Gómez,maria@gmail.com
2,Ana,Martínez,ana@gmail.com
3,Diego,López,diego@gmail.com
4,Sofía,Fernández,sofia@gmail.com
5,Bautista,González,bautista@gmail.com
6,Valentina,Álvarez,valentina@gmail.com
7,Mateo,Romero,mateo@gmail.com
8,Emma,Sánchez,emma@gmail.com
9,Felipe,Benítez,felipe@gmail.com


--------------------------------------------------------------------------------

=== CASO DE PRUEBA #3 ===
❓ PREGUNTA: Listar los autores de nacionalidad Argentina.
💻 SQL GENERADO:
SELECT nombre, apellido FROM autor WHERE nacionalidad = 'Argentina'

📊 RESULTADO:


...

--------------------------------------------------------------------------------

=== CASO DE PRUEBA #10 ===
❓ PREGUNTA: Mostrame los préstamos que están vencidos actualmente.
💻 SQL GENERADO:
SELECT * FROM prestamo WHERE estado = 'VENCIDO'

📊 RESULTADO:


,id_prestamo,id_socio,id_ejemplar,fecha_prestamo,fecha_vencimiento,fecha_devolucion,estado
0,47,3,6,2026-05-01,2026-05-16,None,VENCIDO
1,48,8,9,2026-05-02,2026-05-17,None,VENCIDO
2,49,9,12,2026-05-03,2026-05-18,None,VENCIDO
3,50,10,18,2026-05-05,2026-05-20,None,VENCIDO


--------------------------------------------------------------------------------

=== CASO DE PRUEBA #11 ===
❓ PREGUNTA: Aquellos libros para los que existe más de un ejemplar, tal que al menos dos de esos ejemplares se hayan encontrado prestados en forma simultánea en un determinado momento. Para simplificar, considerar solamente aquellos préstamos en los que el libro ya haya sido devuelto.
💻 SQL GENERADO:
SELECT DISTINCT e1.isbn, l.titulo 
FROM prestamo p1 
JOIN ejemplar e1 ON p1.id_ejemplar = e1.id_ejemplar 
JOIN libro l ON e1.isbn = l.isbn 
JOIN prestamo p2 ON e1.isbn = (SELECT e2.isbn FROM ejemplar e2 WHERE e2.id_ejemplar = p2.id_ejemplar) 
JOIN ejemplar e2 ON p2.id_ejemplar = e2.id_ejemplar 
WHERE p1.id_ejemplar <> p2.id_ejemplar 
AND p1.fecha_devolucion IS NOT NULL 
AND p2.fecha_devolucion IS NOT NULL 
AND (p1.fecha_prestamo <= p2.fecha_devolucion AND p1.fecha_devolucion >= p2.fecha_prestamo) 
AND e1.isbn IN (SELECT isbn FROM ejemplar GROUP BY isbn HAVING COUNT(id_ejemplar) > 1)

,isbn,titulo
0,9780553293357,Fundación
1,9780451457998,2001: Una Odisea Espacial
2,9780261103573,El Señor de los Anillos: La Comunidad del Anillo
3,9780553103540,Juego de Tronos


--------------------------------------------------------------------------------

